# 03 アルゴリズムの比較

- **同じ課題・同じ入力**で、入力サイズを変えて Algorithm A・B を実行する
- **反復測定**して中央値を代表値にする（`time.perf_counter()`）
- **理論計算量**と**実測時間**を区別して比べる

> 初期デモ（**線形探索 vs 二分探索**）は、アルゴリズムを比較する方法を示す**教材例**です。研究テーマを固定するものではありません。チームのテーマに合わせて、`algorithms/` と比較の課題（task）を置き換えてください。

## 準備：実験環境（最初に必ず実行）

F5 リポジトリの `.py`（`algorithms/`・`experiments/`）を import できるように、作業ディレクトリ（F5 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F5 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。

In [ ]:
#@title 準備（F5 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f5-open-research-project"     # リポジトリの中の F5 フォルダ（ルートが F5 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f5_root(p):
    return (Path(p) / "algorithms" / "algorithm_a.py").exists() and (Path(p) / "experiments" / "benchmark.py").exists()


def find_f5_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F5 のルート
        if is_f5_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f5_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F5 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f5_upload")
    for hit in glob.glob("/content/f5_upload/**/algorithms/algorithm_a.py", recursive=True):
        root = Path(hit).parent.parent
        if is_f5_root(root):
            return root
    return None


F5_ROOT = find_f5_root()
if F5_ROOT is None:
    raise RuntimeError("F5 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F5_ROOT)
if str(F5_ROOT) not in sys.path:
    sys.path.insert(0, str(F5_ROOT))

# ---- 既存の .py（アルゴリズム・測定・比較）と、グラフ ----
import platform
import matplotlib.pyplot as plt
from algorithms import algorithm_a as alg_a     # TODO：研究テーマに合わせて置き換える
from algorithms import algorithm_b as alg_b
from experiments import benchmark as bm
from experiments import compare_algorithms as ca
from experiments import plots

plots.setup_style()                    # 文字を大きくするなら plots.setup_style(1.3)
SAVE_FIGURES = False                   # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", plots.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns=ca.TABLE_COLUMNS):
    """比較結果の表。pandas があれば DataFrame、なければ文字の表（時間の列 _s はミリ秒）"""
    try:
        import pandas as pd
        from IPython.display import display
        data = [{h: (None if r.get(k) is None else (r[k] * 1e3 if k.endswith("_s") else r[k])) for k, h, _f in columns}
                for r in rows]
        display(pd.DataFrame(data))
    except ImportError:
        print(ca.format_table(rows, columns))


print("F5_ROOT:", F5_ROOT)
print("実行環境:", ca.environment(), "Colab" if IN_COLAB else "local")
print("Algorithm A:", alg_a.NAME, "/ Algorithm B:", alg_b.NAME)

## 1. 実験条件

| 測る時間 | 意味 | この Notebook での扱い |
| --- | --- | --- |
| データ生成時間 | 入力データを作る時間 | 比較には使わない（`gen_s`） |
| 前処理時間 | 例：二分探索のためのソート | `prep_s`（中央値） |
| アルゴリズム実行時間 | 探索 `QUERIES` 回分の時間 | `run_s`（中央値）。短い処理は 1 回の測定で何回も呼び出す |
| 結果検証時間 | 結果が正しいかの確認 | 測定値に含めない（`verify_s`） |

**前処理を含めない比較**（Run）と、**含める比較**（Prep + Run）を区別します。

In [ ]:
SIZES = [100, 1_000, 10_000]      # TODO：入力サイズを変更する
REPEAT = 5                        # TODO：反復回数を変更する
WARMUP = 1
QUERIES = 100                     # 1 回の測定で探す値の数
HIT_RATIO = 0.5                   # TODO：入力データの特徴を変更する（0 = 全部見つからない = 線形探索の最悪ケース）
ORDER = "random"                  # "random"・"sorted"・"reversed"
SEED = 0

task = ca.make_search_task(QUERIES, HIT_RATIO, ORDER)
rows = ca.run_comparison(task, SIZES, REPEAT, WARMUP, SEED)
print("課題:", task["description"])
print("全行が正しく、A と B の答えが一致:", all(r["correct"] and r["consistent"] for r in rows))
table(rows)

## 2. 入力サイズと実行時間

両対数グラフでは、$O(N)$ は傾き 1、$O(\log N)$ はほぼ水平に近い線になります。破線は「前処理（ソート）+ 実行」です。

In [ ]:
#@title 入力サイズと実行時間
show(plots.plot_time_vs_size(rows), "03_time_vs_size")

## 3. Algorithm A と B の比較（入力サイズごと）

誤差棒は、反復測定の最小〜最大です。

In [ ]:
#@title A と B の比較
show(plots.plot_bars_by_size(rows), "03_bars")
show(plots.plot_bars_by_size(rows, "Including preprocessing: prep + run (measured)", value="total_s"), "03_bars_total")

## 4. 理論計算量の違い

理論計算量は「N が増えたときの増え方」を表します。秒数そのものではありません。最小の N で 1 にそろえて比べます。

In [ ]:
#@title 理論計算量と実測
COMPLEXITY = {"A": "O(N)", "B": "O(log N)"}      # TODO：チームのアルゴリズムの計算量にする
show(plots.plot_theory(SIZES, {alg_a.NAME: "O(N)", alg_b.NAME: "O(log N)", "sort (preprocess)": "O(N log N)"}),
     "03_theory")
show(plots.plot_measured_vs_theory(rows, COMPLEXITY), "03_measured_vs_theory")

## 5. 結果の品質・実行環境によらない指標

探索では答え（見つかったか）は同じなので、品質の代わりに**比較回数**を比べます。
DFS と BFS のように答えの品質が違う課題では、経路長などの品質も比べてください。

In [ ]:
#@title 比較回数
show(plots.plot_operations(rows), "03_operations")

## 6. 反復測定のばらつき

同じ条件でも、測定値は毎回少しずつ違います（他のプログラム・キャッシュ・CPU の周波数など）。だから中央値を使います。

In [ ]:
#@title 反復測定のばらつき
show(plots.plot_repeats(rows, SIZES[-1]), "03_repeats")

## 7. 学生向け TODO

In [ ]:
# TODO 1：入力サイズを変更する（例：[1_000, 5_000, 20_000, 50_000]）
# TODO 2：反復回数を変更する（REPEAT = 3、10 で中央値はどれくらい変わるか）
# TODO 3：入力データの特徴を変更する（HIT_RATIO = 0.0 で最悪ケース、ORDER = "sorted"）
worst = ca.run_comparison(ca.make_search_task(QUERIES, hit_ratio=0.0), [100, 1_000], REPEAT, WARMUP, SEED)
table(worst)

In [ ]:
# TODO 4：アルゴリズムを追加する（例：Python の list.index を基準として加える）
#   ※ ライブラリ関数は比較の基準にはできるが、それだけでは「アルゴリズムの実装」にならない
def run_builtin(_prepared, inp):
    out = []
    for k in inp["queries"]:
        try:
            out.append(inp["data"].index(k))
        except ValueError:
            out.append(-1)
    return out

task3 = ca.make_search_task(QUERIES, HIT_RATIO, ORDER)
task3["algorithms"].append({"key": "C", "label": "C: list.index (built-in)", "prepare": None,
                            "run": run_builtin, "count": None})
rows3 = ca.run_comparison(task3, SIZES, REPEAT, WARMUP, SEED)
table(rows3)
show(plots.plot_time_vs_size(rows3, "A vs B vs built-in (measured)"), "03_three_algorithms")

In [ ]:
# TODO 5：結果を CSV に保存し、docs/experiment_log.md に実行環境と一緒に記録する
# ca.save_csv(rows, "results/search_comparison.csv")
print(ca.environment())

## 考察（チームで記入）

- **TODO**：実測時間の増え方は、理論計算量の予想と合っていたか。合わない部分があれば、その理由は何か
- **TODO**：前処理を含めると、結論はどう変わったか
- **TODO**：測定値のばらつきは、A と B の差と比べて大きいか
- **TODO**：それぞれのアルゴリズムの長所・短所

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（初期デモ：第3章 線形探索・2 分探索）
- F5 リポジトリの `README.md`（計算量、実行時間の測定、正しさの検証）、`textbook/README.md`（教科書コードの一覧と参照方法）
- TODO：チームが参考にした文献・データの出典を追加する